In [ ]:
# Code for BirdCLEF 2026 competition

# In this version, I replaced my previous PCA and dimensionality reduction with a more 
# straightforward approach using the mean and standard deviation of the Mel spectrogram features.

# I wrote this code in Python 3.11.9 because later versions are incompatible with tensorflow.

!pip install librosa

In [64]:
!pip install matplotlib

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 98.7 MB/s  0:00:00
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 102.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 115.5 MB/s  0:00:00
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [matplotlib]7 [matplotlib]


In [1]:
import librosa
import numpy as np
import pandas as pd
import os

In [2]:
# This loop counts the number of audio files in the training directory.

train_soundscapes_labels_df = pd.read_csv("train_soundscapes_labels.csv")

# We turn each 5 second increment into a separate row in the dataframe, with the corresponding labels.
# We have 1,478 files. Given that we have more than 1,000 files, we use spectograms and a CNN.
# Appends the start time to each filename.
# Because each file is 5 seconds long, we don't need to append the end time.

train_soundscapes_reduced_df = pd.DataFrame({
    "filename": train_soundscapes_labels_df.iloc[:, :2].astype(str).agg(''.join, axis=1),
    "primary_label": train_soundscapes_labels_df.iloc[:, 3]
})

print("Length of reduced dataframe:", len(train_soundscapes_reduced_df))

Length of reduced dataframe: 1478


In [3]:
# For each file, we list all animals that are present in the file
# We then classify the file as belonging to the class of each of those animals.

taxonomy_df = pd.read_csv("taxonomy.csv")
classes = ['Insecta', 'Reptilia', 'Amphibia', 'Mammalia', 'Aves']
filenames = {value: [] for value in classes}

for entry in train_soundscapes_reduced_df.itertuples():
    filename = entry.filename
    primary_label = entry.primary_label.split(';') # This records all species in a file.

    found = {value: False for value in classes}
    
    for label in primary_label:
        for value in classes:
            if taxonomy_df[taxonomy_df['primary_label'] == label]['class_name'].values[0] == value:
                found[value] = True
    
    for value in classes:
         if found[value]:
            filenames[value].append(filename)

In [4]:
# This block counts the number of files featuring each class of animal.
# It is troubling that only 26 of the files have reptiles.

print(f"Insecta: {len(filenames['Insecta'])}")
print(f"Reptilia: {len(filenames['Reptilia'])}")
print(f"Amphibia: {len(filenames['Amphibia'])}")
print(f"Mammalia: {len(filenames['Mammalia'])}")
print(f"Aves: {len(filenames['Aves'])}")

Insecta: 336
Reptilia: 26
Amphibia: 1132
Mammalia: 82
Aves: 424


In [ ]:
# This code trains a CNN to classify audio files in OGG format.
# It loads the audio, converts it to Mel spectrograms, and trains a model on the resulting features.
# Make sure to install the required libraries and adjust the DATA_DIR path to your dataset.

# As a test run, I am only using the files that feature insects.

import tensorflow
from tensorflow.keras import layers, models
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

X = []
y = []

N_MELS = 128 # This is the number of Mel bands to generate. It is the standard amount.
SR = 32000 # This is the sample rate.

for filename in train_soundscapes_reduced_df['filename']:
    
    # For each file, we load the corresponding audio segment based on the filename.
    # We extract the start time from the filename and load the 5-second audio segment
    # starting from that time.
    # power_to_db converts the Mel spectrogram to decibel units,
    # which is a common preprocessing step for audio data.
    
    file = filename[:-8]
    start = int(filename[-2:])
    audio, sample_rate = librosa.load(f"train_soundscapes/{file}", sr=None, offset=start, duration=5)
    
    # Convert to Mel spectrogram
    
    # power_to_db converts the Mel spectrogram to decibel units,
    # which is a common preprocessing step for audio data.
    
    mel_spec = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=N_MELS)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    # Normalize
    mel_spec_db = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min())
    
    # Add channel dimension
    # Makes data compatible with CNN input requirements. The shape becomes (n_mels, time_steps, 1).
    X.append(mel_spec_db[..., np.newaxis])
    if filename in filenames['Insecta']:
        y.append('Insecta')
    else:
        y.append('Other')
    
X = np.array(X, dtype=np.float32)
y = np.array(y)

# Encode labels

le = LabelEncoder()
y_encoded = le.fit_transform(y) # Converts 'Insecta' to 0 and 'Other' to 1.

# Train/test split

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

In [20]:
# Build CNN model

# padding = 'same' means that the output has the same dimensions as the input.
# It accomplishes this by adding zero-padding to the input as needed.

# (3, 3) is the kernel size. It defines the size of the filter that will be applied to the input data.
# We reduce to a 64 x 64 feature map in the final convolutional layer,
# which is a common choice for audio classification tasks.

def build_model(input_shape=(N_MELS, X.shape[2], 1), num_classes=2):
    model = models.Sequential()

    # Block 1
    model.add(layers.Conv2D(16, (3, 3), activation='relu', padding='same', input_shape=input_shape))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    # Block 2
    model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    # Block 3
    model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    # Global pooling
    model.add(layers.GlobalAveragePooling2D())

    # Dense head
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(0.4))

    # Output layer
    model.add(layers.Dense(num_classes, activation='softmax'))

    return model

model = build_model()
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# -----------------------------
# 6️⃣ Train the model
# -----------------------------
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    batch_size=32,
    epochs=20
)

Epoch 1/20


/Users/noah/tf_env/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


37/37 ━━━━━━━━━━━━━━━━━━━━ 8s 199ms/step - accuracy: 0.8976 - loss: 0.2697 - val_accuracy: 0.1993 - val_loss: 0.7527
Epoch 2/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 181ms/step - accuracy: 0.9602 - loss: 0.1435 - val_accuracy: 0.1993 - val_loss: 0.9688
Epoch 3/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 181ms/step - accuracy: 0.9569 - loss: 0.1322 - val_accuracy: 0.1993 - val_loss: 0.9171
Epoch 4/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 179ms/step - accuracy: 0.9611 - loss: 0.1180 - val_accuracy: 0.8007 - val_loss: 0.6209
Epoch 5/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 180ms/step - accuracy: 0.9797 - loss: 0.0771 - val_accuracy: 0.8007 - val_loss: 0.5124
Epoch 6/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 180ms/step - accuracy: 0.9772 - loss: 0.0652 - val_accuracy: 0.8007 - val_loss: 0.6474
Epoch 7/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 181ms/step - accuracy: 0.9797 - loss: 0.0601 - val_accuracy: 0.8007 - val_loss: 0.9097
Epoch 8/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 181ms/step - accuracy: 0.9848 - loss: 0.0462 - val_accuracy: 0.8007 - val_

In [ ]:
# accuracy = 99.58% on the test set.
# val_accuracy = 78.38% on the validation set. The discrepancy is due to overfitting.
# This version has 98.31% val_accuracy, which is a significant improvement. The model is less overfit.

# In this block, we use early stopping.

from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 185ms/step - accuracy: 0.9932 - loss: 0.0248 - val_accuracy: 0.9426 - val_loss: 0.1251
Epoch 2/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 180ms/step - accuracy: 0.9932 - loss: 0.0179 - val_accuracy: 1.0000 - val_loss: 0.0130
Epoch 3/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 181ms/step - accuracy: 0.9958 - loss: 0.0119 - val_accuracy: 0.9595 - val_loss: 0.0777
Epoch 4/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 180ms/step - accuracy: 0.9949 - loss: 0.0146 - val_accuracy: 0.9865 - val_loss: 0.0313
Epoch 5/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 179ms/step - accuracy: 1.0000 - loss: 0.0078 - val_accuracy: 0.9764 - val_loss: 0.1161
Epoch 6/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 180ms/step - accuracy: 0.9975 - loss: 0.0097 - val_accuracy: 0.9493 - val_loss: 0.1348
Epoch 7/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 181ms/step - accuracy: 0.9983 - loss: 0.0074 - val_accuracy: 1.0000 - val_loss: 0.0032
Epoch 8/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 183ms/step - accuracy: 0.9983 - loss: 0.0075 - val_accuracy: 0.

In [ ]:
# As a check, here is a version with small convolutional layers and no early stopping. It is overfitting, but it is a check that the code runs.

def build_model(input_shape=(N_MELS, X.shape[2], 1), num_classes=2):
    model = models.Sequential()

    # Block 1
    model.add(layers.Conv2D(8, (3, 3), activation='relu', padding='same', input_shape=input_shape))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    # Block 2
    model.add(layers.Conv2D(16, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    # Block 3
    model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    # Global pooling
    model.add(layers.GlobalAveragePooling2D())

    # Dense head
    model.add(layers.Dense(32, activation='relu'))
    model.add(layers.Dropout(0.4))

    # Output layer
    model.add(layers.Dense(num_classes, activation='softmax'))

    return model

model = build_model()
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# -----------------------------
# Train the model
# -----------------------------
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    batch_size=32,
    epochs=20
)

Epoch 1/20


/Users/noah/tf_env/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 101ms/step - accuracy: 0.8536 - loss: 0.3197 - val_accuracy: 0.6689 - val_loss: 0.6691
Epoch 2/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 97ms/step - accuracy: 0.9188 - loss: 0.2396 - val_accuracy: 0.2635 - val_loss: 0.7558
Epoch 3/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 97ms/step - accuracy: 0.9484 - loss: 0.1779 - val_accuracy: 0.2872 - val_loss: 0.7901
Epoch 4/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 97ms/step - accuracy: 0.9475 - loss: 0.1636 - val_accuracy: 0.5507 - val_loss: 0.6551
Epoch 5/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 98ms/step - accuracy: 0.9450 - loss: 0.1528 - val_accuracy: 0.7568 - val_loss: 0.5541
Epoch 6/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 98ms/step - accuracy: 0.9628 - loss: 0.1237 - val_accuracy: 0.7736 - val_loss: 0.5702
Epoch 7/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 103ms/step - accuracy: 0.9687 - loss: 0.0930 - val_accuracy: 0.7905 - val_loss: 0.5723
Epoch 8/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 101ms/step - accuracy: 0.9645 - loss: 0.1129 - val_accuracy: 0.8007 - val_loss:

In [ ]:
# That code had 98.65% value accuracy, which is a slight improvement over the previous version.
# The model is still overfitting, but it is a check that the code runs.

# Here is a version with smaller convolutional layers and early stopping.
# The value accuracy is very bad! This is likely because the model is too small to learn the task.
# However, it is also possible that early stopping is preventing the model from learning.
# We will need to experiment with the patience parameter to find the optimal value.

def build_model(input_shape=(N_MELS, X.shape[2], 1), num_classes=2):
    model = models.Sequential()

    # Block 1
    model.add(layers.Conv2D(8, (3, 3), activation='relu', padding='same', input_shape=input_shape))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    # Block 2
    model.add(layers.Conv2D(16, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    # Block 3
    model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    # Global pooling
    model.add(layers.GlobalAveragePooling2D())

    # Dense head
    model.add(layers.Dense(32, activation='relu'))
    model.add(layers.Dropout(0.4))

    # Output layer
    model.add(layers.Dense(num_classes, activation='softmax'))

    return model

model = build_model()
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 100ms/step - accuracy: 0.8350 - loss: 0.3648 - val_accuracy: 0.1993 - val_loss: 0.7908
Epoch 2/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 98ms/step - accuracy: 0.9171 - loss: 0.2335 - val_accuracy: 0.1993 - val_loss: 0.9801
Epoch 3/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 98ms/step - accuracy: 0.9408 - loss: 0.1787 - val_accuracy: 0.1993 - val_loss: 1.0812
Epoch 4/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 97ms/step - accuracy: 0.9509 - loss: 0.1476 - val_accuracy: 0.1993 - val_loss: 1.1050
Epoch 5/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 97ms/step - accuracy: 0.9619 - loss: 0.1297 - val_accuracy: 0.1993 - val_loss: 0.8586
Epoch 6/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 98ms/step - accuracy: 0.9712 - loss: 0.1016 - val_accuracy: 0.3209 - val_loss: 0.7286
Epoch 7/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 98ms/step - accuracy: 0.9704 - loss: 0.0924 - val_accuracy: 0.7534 - val_loss: 0.6453
Epoch 8/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 99ms/step - accuracy: 0.9687 - loss: 0.0918 - val_accuracy: 0.6588 - 

In [ ]:
# Because the previous version of the model underfit, I am going to relax the early stopping criteria by increasing the patience parameter to 10.
# I also decreased the dropout rate to 0.2.
# This version works really well! It had 100% value accuracy on the last few epochs.

def build_model(input_shape=(N_MELS, X.shape[2], 1), num_classes=2):
    model = models.Sequential()

    # Block 1
    model.add(layers.Conv2D(8, (3, 3), activation='relu', padding='same', input_shape=input_shape))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    # Block 2
    model.add(layers.Conv2D(16, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    # Block 3
    model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    # Global pooling
    model.add(layers.GlobalAveragePooling2D())

    # Dense head
    model.add(layers.Dense(32, activation='relu'))
    model.add(layers.Dropout(0.2))

    # Output layer
    model.add(layers.Dense(num_classes, activation='softmax'))

    return model

model = build_model()
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/50


/Users/noah/tf_env/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 103ms/step - accuracy: 0.8875 - loss: 0.2713 - val_accuracy: 0.6318 - val_loss: 0.6777
Epoch 2/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 98ms/step - accuracy: 0.9357 - loss: 0.1865 - val_accuracy: 0.6588 - val_loss: 0.6663
Epoch 3/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 97ms/step - accuracy: 0.9391 - loss: 0.1725 - val_accuracy: 0.5405 - val_loss: 0.6851
Epoch 4/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 97ms/step - accuracy: 0.9687 - loss: 0.1114 - val_accuracy: 0.8007 - val_loss: 0.6015
Epoch 5/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 97ms/step - accuracy: 0.9729 - loss: 0.0964 - val_accuracy: 0.8007 - val_loss: 0.5541
Epoch 6/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 97ms/step - accuracy: 0.9822 - loss: 0.0704 - val_accuracy: 0.8007 - val_loss: 0.5682
Epoch 7/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 98ms/step - accuracy: 0.9797 - loss: 0.0644 - val_accuracy: 0.7939 - val_loss: 0.6082
Epoch 8/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 97ms/step - accuracy: 0.9856 - loss: 0.0560 - val_accuracy: 0.6892 - val_loss: 0

In [35]:
# In this block, I make analogous models for the other four classes.

from tensorflow.keras import models

N_MELS = 128
SR = 32000

models_class = {}

def build_model(input_shape=(N_MELS, X.shape[2], 1), num_classes=2):
    model = models.Sequential()

    # Block 1
    model.add(layers.Conv2D(8, (3, 3), activation='relu', padding='same', input_shape=input_shape))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    # Block 2
    model.add(layers.Conv2D(16, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    # Block 3
    model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    # Global pooling
    model.add(layers.GlobalAveragePooling2D())

    # Dense head
    model.add(layers.Dense(32, activation='relu'))
    model.add(layers.Dropout(0.2))

    # Output layer
    model.add(layers.Dense(num_classes, activation='softmax'))

    return model

for value in ['Reptilia', 'Amphibia', 'Mammalia', 'Aves']:
    X = []
    y = []

    for filename in train_soundscapes_reduced_df['filename']:
        file = filename[:-8]
        start = int(filename[-2:])
        audio, sample_rate = librosa.load(f"train_soundscapes/{file}", sr=None, offset=start, duration=5)
        mel_spec = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=N_MELS)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        mel_spec_db = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min())
    
        X.append(mel_spec_db[..., np.newaxis])
        if filename in filenames[value]:
            y.append(value)
        else:
            y.append('Other')
    
    X = np.array(X, dtype=np.float32)
    y = np.array(y)
    
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
    
    # We call the build_model function inside the loop to create a new model for each class.
    
    model = build_model()
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    )

    # Because Reptilia is the smallest class, we use a smaller batch size to prevent overfitting.
    
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=50,
        batch_size = 32 - 24*(value == 'Reptilia'), # Batch size is 32 for all classes except Reptilia, which has a batch size of 8.
        callbacks=[early_stop]
    )
    
    models_class[value] = model

Epoch 1/50


/Users/noah/tf_env/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


148/148 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.9501 - loss: 0.1624 - val_accuracy: 0.9831 - val_loss: 0.4550
Epoch 2/50
148/148 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.9822 - loss: 0.0823 - val_accuracy: 0.9831 - val_loss: 0.4798
Epoch 3/50
148/148 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - accuracy: 0.9822 - loss: 0.0824 - val_accuracy: 0.9831 - val_loss: 0.2364
Epoch 4/50
148/148 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - accuracy: 0.9822 - loss: 0.0759 - val_accuracy: 0.9831 - val_loss: 0.0946
Epoch 5/50
148/148 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step - accuracy: 0.9822 - loss: 0.0676 - val_accuracy: 0.9831 - val_loss: 0.0666
Epoch 6/50
148/148 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - accuracy: 0.9822 - loss: 0.0622 - val_accuracy: 0.9831 - val_loss: 0.0613
Epoch 7/50
148/148 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step - accuracy: 0.9822 - loss: 0.0678 - val_accuracy: 0.9831 - val_loss: 0.0846
Epoch 8/50
148/148 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - accuracy: 0.9822 - loss: 0.0597 - val_accuracy: 0.983

/Users/noah/tf_env/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 100ms/step - accuracy: 0.7369 - loss: 0.5378 - val_accuracy: 0.2365 - val_loss: 0.7971
Epoch 2/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 102ms/step - accuracy: 0.8900 - loss: 0.2870 - val_accuracy: 0.2365 - val_loss: 1.0767
Epoch 3/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 107ms/step - accuracy: 0.8993 - loss: 0.2405 - val_accuracy: 0.2365 - val_loss: 1.2728
Epoch 4/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 101ms/step - accuracy: 0.9239 - loss: 0.2076 - val_accuracy: 0.2365 - val_loss: 1.3217
Epoch 5/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 109ms/step - accuracy: 0.9239 - loss: 0.1918 - val_accuracy: 0.2365 - val_loss: 1.3645
Epoch 6/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 103ms/step - accuracy: 0.9323 - loss: 0.1858 - val_accuracy: 0.2365 - val_loss: 1.3443
Epoch 7/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 105ms/step - accuracy: 0.9535 - loss: 0.1514 - val_accuracy: 0.2365 - val_loss: 1.2525
Epoch 8/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 103ms/step - accuracy: 0.9509 - loss: 0.1380 - val_accuracy: 0.2365 - val_

/Users/noah/tf_env/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 104ms/step - accuracy: 0.9509 - loss: 0.2028 - val_accuracy: 0.1081 - val_loss: 0.7113
Epoch 2/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 100ms/step - accuracy: 0.9501 - loss: 0.1779 - val_accuracy: 0.0811 - val_loss: 0.8216
Epoch 3/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 102ms/step - accuracy: 0.9509 - loss: 0.1751 - val_accuracy: 0.0811 - val_loss: 0.8215
Epoch 4/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 100ms/step - accuracy: 0.9509 - loss: 0.1646 - val_accuracy: 0.3953 - val_loss: 0.7417
Epoch 5/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 100ms/step - accuracy: 0.9509 - loss: 0.1608 - val_accuracy: 0.7027 - val_loss: 0.6202
Epoch 6/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 101ms/step - accuracy: 0.9509 - loss: 0.1525 - val_accuracy: 0.7804 - val_loss: 0.5819
Epoch 7/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 100ms/step - accuracy: 0.9526 - loss: 0.1337 - val_accuracy: 0.8682 - val_loss: 0.4949
Epoch 8/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 100ms/step - accuracy: 0.9509 - loss: 0.1387 - val_accuracy: 0.7872 - val_

/Users/noah/tf_env/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 103ms/step - accuracy: 0.7606 - loss: 0.4994 - val_accuracy: 0.4865 - val_loss: 0.6955
Epoch 2/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 100ms/step - accuracy: 0.7758 - loss: 0.4509 - val_accuracy: 0.3142 - val_loss: 0.7276
Epoch 3/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 100ms/step - accuracy: 0.8037 - loss: 0.3954 - val_accuracy: 0.3142 - val_loss: 0.7267
Epoch 4/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 100ms/step - accuracy: 0.8409 - loss: 0.3626 - val_accuracy: 0.6284 - val_loss: 0.6903
Epoch 5/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 101ms/step - accuracy: 0.8629 - loss: 0.3218 - val_accuracy: 0.6858 - val_loss: 0.6399
Epoch 6/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 100ms/step - accuracy: 0.8849 - loss: 0.2778 - val_accuracy: 0.6858 - val_loss: 0.6121
Epoch 7/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 102ms/step - accuracy: 0.8900 - loss: 0.2567 - val_accuracy: 0.6858 - val_loss: 0.6215
Epoch 8/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 100ms/step - accuracy: 0.8993 - loss: 0.2389 - val_accuracy: 0.6858 - val_

In [40]:
# In this code block, we evaluate the models on the test set and print the results.

for value in ['Reptilia', 'Amphibia', 'Mammalia', 'Aves']:
    
    model = models_class[value]
    true_positive = 0
    true_negative = 0
    false_positive = 0
    false_negative = 0
    
    for filename in train_soundscapes_reduced_df['filename']:
        
        file = filename[:-8]
        start = int(filename[-2:])
        audio, sample_rate = librosa.load(f"train_soundscapes/{file}", sr=None, offset=start, duration=5)
        mel_spec = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=N_MELS)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        mel_spec_db = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min())
        
        pred = model.predict(mel_spec_db[np.newaxis, ..., np.newaxis], verbose=0)[0][0]
        
        if filename in filenames[value]:
            if pred > 0.5:
                true_positive += 1
            else:
                false_negative += 1
        else:
            if pred > 0.5:
                false_positive += 1
            else:
                true_negative += 1
        
    print(f"{value} - TP: {true_positive}, TN: {true_negative}, FP: {false_positive}, FN: {false_negative}")


Reptilia - TP: 26, TN: 0, FP: 1452, FN: 0
Amphibia - TP: 0, TN: 346, FP: 0, FN: 1132
Mammalia - TP: 20, TN: 1386, FP: 10, FN: 62
Aves - TP: 406, TN: 1016, FP: 38, FN: 18


In [71]:
# The code works very well for Mammalia and Aves, but it performs poorly for Reptilia and Amphibia.
# This is likely because there are very few files featuring Reptilia or not featuring Amphibia.
# Also, for Reptilia, it only recorded positives and for Amphibia, it only recorded negatives.

# In this code block, we make a new model for Reptilia.

from tensorflow.keras import models
from sklearn.utils.class_weight import compute_class_weight

N_MELS = 128
SR = 32000

def build_model(input_shape=(N_MELS, X.shape[2], 1), num_classes=2):
    model = models.Sequential()

    # Block 1
    model.add(layers.Conv2D(8, (3, 3), activation='relu', padding='same', input_shape=input_shape))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    # Block 2
    model.add(layers.Conv2D(16, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    # Block 3
    model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    # Global pooling
    model.add(layers.GlobalAveragePooling2D())

    # Dense head
    model.add(layers.Dense(32, activation='relu'))
    model.add(layers.Dropout(0.2))

    # Output layer
    model.add(layers.Dense(num_classes, activation='softmax'))

    return model

X = []
y = []

for filename in train_soundscapes_reduced_df['filename']:
    file = filename[:-8]
    start = int(filename[-2:])
    audio, sample_rate = librosa.load(f"train_soundscapes/{file}", sr=None, offset=start, duration=5)
    mel_spec = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=N_MELS)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    mel_spec_db = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min())
    
    X.append(mel_spec_db[..., np.newaxis])
    if filename in filenames['Reptilia']:
        y.append(1)
    else:
        y.append(0)

X = np.array(X, dtype=np.float32)
y = np.array(y)
    
le = LabelEncoder()
y_encoded = le.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# This part computes class weights to address the class imbalance in the Reptilia dataset.

class_weights = {1: 1, 0: 5}
    
# We call the build_model function inside the loop to create a new model for each class.
    
model = build_model()
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)
    
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size = 8,
    callbacks=[early_stop],
    class_weight=class_weights
)
    
model_Reptilia = model

KeyboardInterrupt: 

In [70]:
y_probs = model.predict(X_test, verbose=0).flatten()

from sklearn.metrics import precision_recall_curve, f1_score
import matplotlib.pyplot as plt
import numpy as np

thresholds = np.arange(0.1, 1.0, 0.05)
f1_scores = []

for t in thresholds:
    y_pred = (y_probs > t).astype(int)
    for value in y_pred:
        if value == 1:
            value = 'Reptilia'
        else:
            value = 'Other'
    f1 = f1_score(y_test, y_pred)
    f1_scores.append(f1)

# Find threshold with max F1
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
print(f"Best F1 score = {f1_scores[best_idx]:.3f} at threshold = {best_threshold:.2f}")

ValueError: Found input variables with inconsistent numbers of samples: [296, 592]

In [68]:
y_probs = model.predict(X_test, verbose=0).flatten()
print(y_probs[:5])

[1.0000000e+00 8.3668206e-10 9.9999404e-01 5.9982185e-06 9.8646885e-01]
